# 08 · Validación final, sesgos y robustez (Fases 11 y 13)

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 30)
from src.config import *
df = pd.read_parquet(PROCESSED / "emovi_movilidad.parquet")
q1 = df[df.sigue_q1.notna()]
from src.models import train as T
tr, te = T.split(df)
tabla, modelos = T.compare(tr, te, T.NUM, T.CAT)
elegido, mejor_auc = T.choose(tabla)
modelo = modelos[elegido]
from src.features.indices import weighted_quintile

## 1. Robustez: ¿el hallazgo principal depende de cómo se mide?

In [2]:
w = df.factor
def tasa(qo, qd, m=None):
    m = (qo == 1) if m is None else m
    return np.average(qd[m] == 1, weights=w[m]) * 100
qo_all = weighted_quintile(df.indice_origen, w); qd_all = weighted_quintile(df.indice_destino, w)
qi = pd.Series(0, index=df.index)
for _, g in df.groupby("cohorte"): qi[g.index] = weighted_quintile(g.ingreso_pc, g.factor)
rng = np.random.default_rng(SEED); x = q1.sigue_q1.values; ww = q1.factor.values
boot = [np.average(x[i], weights=ww[i]) * 100 for i in (rng.integers(0, len(x), len(x)) for _ in range(1000))]
rob = pd.DataFrame({"Definición": ["Principal: índice de bienes, quintiles por cohorte", "Quintiles sin separar por cohorte", "Destino medido con ingreso per cápita"],
                    "% sigue en Q1": [tasa(df.q_origen, df.q_destino), tasa(qo_all, qd_all), tasa(df.q_origen, qi)]}).round(1)
rob.to_csv(TABLES / "robustez.csv", index=False); display(rob)
print(f"IC 95% bootstrap (definición principal): {np.percentile(boot, 2.5):.1f}% – {np.percentile(boot, 97.5):.1f}%")

,Definición,% sigue en Q1
0,"Principal: índice de bienes, quintiles por coh...",51.1
1,Quintiles sin separar por cohorte,48.9
2,Destino medido con ingreso per cápita,46.6


IC 95% bootstrap (definición principal): 47.6% – 54.5%


**Conclusión:** en todas las definiciones, entre 47% y 51% de quienes nacen en el 20% más pobre siguen ahí. El hallazgo es robusto. El orden de magnitud es consistente con lo que reporta el CEEY (la cifra exacta se debe verificar en su informe; ver `docs/report/referencias.md`).

## 2. Desempeño por subgrupo (sesgo del modelo)

In [3]:
sub = T.subgroup_performance(modelo, te, T.NUM, T.CAT)
sub["Brecha (real - predicha)"] = (sub["Tasa real sigue Q1"] - sub["Tasa predicha"]).round(3)
sub.to_csv(TABLES / "desempeno_subgrupos.csv", index=False); display(sub)

,Variable,Grupo,n,Tasa real sigue Q1,Tasa predicha,AUC,Brecha (real - predicha)
0,mujer,Hombre,212,0.486,0.495,0.705,-0.009
1,mujer,Mujer,323,0.539,0.522,0.722,0.017
2,tono_piel,Claro (H-K),92,0.370,0.469,0.761,-0.099
3,tono_piel,Medio (F-G),329,0.526,0.527,0.728,-0.001
4,tono_piel,Oscuro (A-E),114,0.614,0.502,0.650,0.112
5,region_14,Centro,123,0.480,0.458,0.585,0.022
6,region_14,Centro-norte,53,0.340,0.312,0.483,0.028
7,region_14,Norte,33,0.273,0.379,0.634,-0.106
8,region_14,Norte-occidente,64,0.359,0.316,0.749,0.043
9,region_14,Sur,261,0.644,0.641,0.713,0.003


**Hallazgo de sesgo:** en el conjunto de prueba (114 personas de tono oscuro), el modelo **subestima** su persistencia: la tasa real es ≈61% y la predicha ≈50%. Si este modelo se usara para focalizar programas, dejaría fuera a personas de tono oscuro que sí siguen en la pobreza. Las tasas de la región Norte y de las ciudades grandes también se desvían, pero ahí la n es pequeña (< 40).

## 3. Red team (Fase 11): riesgos clasificados
| # | Riesgo | Nivel | Evidencia | Acción |
|---|---|---|---|---|
| 1 | **Causalidad injustificada** en los escenarios | **ALTO** | Escolaridad y empleo se miden al mismo tiempo que el destino | Lenguaje condicional obligatorio; se rechazó el escenario de choque adverso |
| 2 | **Modelo sesgado por tono de piel** | **ALTO** | Brecha de calibración de ≈11 pts, presente en **todos** los modelos | Se prohíbe usar el modelo para focalizar; se reporta como hallazgo |
| 3 | **Decisiones tomadas después de ver resultados** (D16, D17) | **MEDIO** | Agrupar la escolaridad de los padres y quitar un escenario | Se justificaron por tamaño de muestra, no por resultados; se dejaron registradas |
| 4 | **Efecto de cohorte como artefacto** | **MEDIO** | Bienes diferentes entre generaciones | No se afirma que la movilidad empeoró |
| 5 | **Sesgo de memoria** (origen retrospectivo) | MEDIO | — | Validación con la escalera subjetiva (r ≈ 0.5); se usan bienes, no ingreso |
| 6 | **Sesgo de supervivencia** | MEDIO | La muestra va de 25 a 64 años: quienes murieron o emigraron no están | Limitación declarada; puede subestimar la persistencia si la pobreza aumenta la mortalidad o la emigración |
| 7 | **Falacia ecológica** (contexto estatal) | MEDIO | Pobreza 2022 vs origen de décadas atrás | Solo descriptivo; ninguna inferencia individual |
| 8 | **Missingness no aleatoria** | BAJO | El padre ausente se asocia con más persistencia | Categoría propia, sin imputación |
| 9 | **Leakage** | BAJO | Revisado: sin variables de destino; imputación dentro del pipeline | — |
| 10 | **Desbalance** | BAJO | 52/48 | Sin efecto (`class_weight` no cambia nada) |
| 11 | **Confusión entre movilidad absoluta y relativa** | BAJO | Se mide movilidad **relativa** (quintiles) | Declarado: que alguien "siga en Q1" no significa que viva igual que sus padres |
| 12 | **Confusión entre educación y oportunidades laborales** | MEDIO | La educación y el empleo formal están correlacionados | Se separaron en los escenarios de la Fase 10 |
| 13 | **Promedios que ocultan heterogeneidad** | MEDIO | La brecha por tono solo existe fuera del Sur (Simpson) | Reportado en 03 y 06 |
| 14 | **Visualizaciones engañosas** | BAJO | Todas las barras empiezan en 0 y muestran la n | — |

**¿Qué afirmación sería la más fácil de atacar académicamente?**
*"La educación es la principal vía de salida de la pobreza."* La escolaridad propia se mide al mismo tiempo que el destino, puede ser consecuencia de recursos no observados (causalidad inversa y variables omitidas) y la H003 no encontró evidencia de que su asociación varíe por región. Debe presentarse como **asociación mediadora**, nunca como efecto causal.

## 4. Checklist técnico final (Fase 13)
| Requisito | Estado |
|---|---|
| Ambiente reproducible documentado | ✅ `requirements.txt` |
| Semillas registradas | ✅ 42 (más una prueba con 10 semillas) |
| Datos originales preservados | ✅ zips en el repo; `data/raw` no se modifica |
| Transformaciones reproducibles | ✅ `src/` + notebooks 01-02 |
| Separación correcta de datos | ✅ 80/20 estratificado + CV de 5 particiones |
| Ausencia de leakage revisada | ✅ |
| Modelos y parámetros registrados | ✅ `src/models/train.py`, `docs/registro_experimentos.md` |
| Métricas reproducibles | ✅ `reports/tables/` |
| Figuras exportadas con títulos y fuentes | ✅ `reports/figures/` |
| Referencias completas | ⚠️ `docs/report/referencias.md`: falta verificar las cifras exactas del CEEY |
| Limitaciones visibles | ✅ |
| Resultados contradictorios documentados | ✅ (tono de piel, madre, H003) |
| Reporte PDF final | ❌ Pendiente (Fase 12) |
| Notebooks o scripts organizados | ✅ |
| README actualizado | ✅ |

## 5. Reproducibilidad
1. Descomprimir los datos en `data/raw/` (ver `docs/base_de_datos.md`).
2. `pip install -r requirements.txt`
3. Ejecutar los notebooks del 01 al 08 en orden.

**Estado de la fase:** PROPUESTA — falta la revisión humana (regla de avance del roadmap: APROBADA / APROBADA CON CAMBIOS / REABRIR / BLOQUEADA).